# Test Time Compute

This notebook demonstrates how to use the Search-and-Learn framework for a single instruction.

In [ ]:
!git clone https://github.com/yogendrahexo/search-and-learn.git
%cd search-and-learn
!git checkout test
!pip install -e .[dev]

In [ ]:
import logging
import torch
from vllm import LLM

from sal.config import Config
from sal.models.reward_models import load_prm
from sal.search import beam_search, best_of_n, dvts

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

## Configurations

In [14]:
# Global configurations and problem definition
APPROACHES = {
    "beam_search": beam_search,
    "dvts": dvts,
    "best_of_n": best_of_n,
}

# Configuration
CONFIG = {
    "model_path": "meta-llama/Llama-3.2-1B-Instruct",
    "temperature": 0.1,
    "approach": "best_of_n",
    "n": 4,
    "search_batch_size": 4,
    "sort_completed": True,
    "filter_duplicates": True,
    "seed": 0,
    "gpu_memory_utilization": 0.3,

    # Available PRMs
    # "Skywork/Skywork-o1-Open-PRM-Qwen-2.5-1.5B",
    # "peiyi9979/math-shepherd-mistral-7b-prm",
    # "RLHFlow/Llama3.1-8B-PRM-Deepseek-Data",
    # "Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B",
    "prm_path": "Skywork/Skywork-o1-Open-PRM-Qwen-2.5-1.5B",

    # for beam search and dvts
    "num_iterations": 5,
    "beam_width": 4,
}

# Problem to solve
PROBLEM = "A bakery sells 12 chocolate cupcakes and 18 vanilla cupcakes. If they sell half of the chocolate cupcakes and 2/3 of the vanilla cupcakes, how many cupcakes are left?"

## Run Experiment

In [ ]:
def process_single_problem(problem, config):
    approach_fn = APPROACHES[config.approach]

    num_gpus = torch.cuda.device_count()
    llm = LLM(
        model=config.model_path,
        gpu_memory_utilization=config.gpu_memory_utilization,
        enable_prefix_caching=True,
        seed=config.seed,
        tensor_parallel_size=num_gpus,

        # if t4
        # dtype="float16",
        # max_model_len=2048,
    )
    prm = load_prm(config)

    # Create examples dict exactly like the dataset structure
    examples = {
        "problem": [problem],
        "solution": [""],
        "answer": [""],
        "subject": ["general"],
        "level": [1],
        "unique_id": ["custom_prompt_1"]
    }
    
    results = approach_fn(examples, config=config, llm=llm, prm=prm)
    torch.cuda.empty_cache()
    return results

# Create config object from dictionary
config = Config()
for key, value in CONFIG.items():
    setattr(config, key, value)

# Process the instruction
results = process_single_problem(PROBLEM, config)

## Print results

In [ ]:
print("\n=== Generated Solutions ===")
for idx, completion in enumerate(results["completions"][0]):
    print(f"\n[Solution {idx + 1}]")
    print("-" * 80)
    print(f"{completion}")
    print("-" * 80)
    
    # Print scores with step information
    scores = results["scores"][0][idx]
    final_score = scores[0]  # First score is used for ranking
    
    print("│ Step-by-Step Scores:")
    for step_idx, score in enumerate(scores, 1):
        print(f"│ Step {step_idx}: {score:.4f}")
    print("│")
    print(f"│ Ranking Score: {final_score:.4f}")
    print("-" * 80)

# Find which solution was chosen as best
best_idx = results["completions"][0].index(results["pred"][0])
best_scores = results["scores"][0][best_idx]

print("\n=== Best Solution Selected ===")
print(f"Solution #{best_idx + 1}")
print("-" * 80)
print(results["pred"][0])
print("-" * 80)
print("│ Step-by-Step Scores:")
for step_idx, score in enumerate(best_scores, 1):
    print(f"│ Step {step_idx}: {score:.4f}")
print("│")
print(f"│ Ranking Score: {best_scores[0]:.4f}")
print("-" * 80)